In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset
import re
dataset = load_dataset("Jaiccc/model0_boundary_predict_streaming", split="train")

train.jsonl:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1374 [00:00<?, ? examples/s]

**Display a particular sample's input context and model's prediction on that sample**

In [ ]:
sample_number = 27
sample = dataset[sample_number]
print("Dataset columns available:", sample.keys())
instruction = sample['instruction']
input_data = sample['input']
expected_output = sample['output']

# Combine the instruction and input into a single user message
combined_user_text = f"{instruction}\n\n{input_data}"

# Format the prompt to match the ChatML-style markers
prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"
inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

# Decode the raw output
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# Extract the assistant block using the exact marker tokens
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
raw_assistant = m.group(1).strip()
print(f"=== ROW {sample_number} COMBINED INPUT ===")
print(combined_user_text)
print("\n=== EXPECTED OUTPUT (From Dataset) ===")
print(expected_output)
print("\n=== MODEL GENERATED OUTPUT ===")
print(raw_assistant)

Dataset columns available: dict_keys(['instruction', 'input', 'output'])
=== ROW 27 COMBINED INPUT ===
Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract 

**Performs a comprehensive evaluation by iterating through every sample in the dataset to validate the model's predictions against the predefined ground truth. For each entry, it compares the assistant's classification to the expected label and calculates a final accuracy metric to quantify the model's overall performance.**

In [ ]:
# 1. Initialize counters for a quick summary
total_samples = len(dataset)
total_samples = 200
correct_count = 0

print(f"Scanning {total_samples} samples...\n")

for i in range(200):
    sample = dataset[i]

    # 2. Extract components
    instruction = sample['instruction']
    input_data = sample['input']
    expected_output = sample['output'].strip()

    # 3. Format the prompt
    combined_user_text = f"{instruction}\n\n{input_data}"
    prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

    # 4. Generate
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=64, # Classification is short, 64 is plenty
        use_cache=True,
        temperature=0.1,   # Lower temperature for more consistent classification
    )

    # 5. Decode and Clean
    raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

    # Extract only the assistant's specific response
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
    model_result = m.group(1).strip() if m else raw_output.split("assistant")[-1].strip()

    # 6. Compare and Print
    is_correct = "✅" if model_result == expected_output else "❌"
    if model_result == expected_output:
        correct_count += 1

    print(f"--- Sample {i} ---")
    print(f"EXPECTED: {expected_output}")
    print(f"MODEL:    {model_result} {is_correct}")
    print("-" * 20)

# 7. Final Summary
accuracy = (correct_count / total_samples) * 100
print(f"\nFinal Accuracy: {accuracy:.2f}% ({correct_count}/{total_samples})")

Scanning 200 samples...

--- Sample 0 ---
EXPECTED: 8.45968, new event
MODEL:    8.45968, new event ✅
--------------------
--- Sample 1 ---
EXPECTED: 39.014894, new event
MODEL:    39.014894, old event ❌
--------------------
--- Sample 2 ---
EXPECTED: 39.950857, new event
MODEL:    39.950857, new event ✅
--------------------
--- Sample 3 ---
EXPECTED: 60.285293, new event
MODEL:    60.285293, old event ❌
--------------------
--- Sample 4 ---
EXPECTED: 64.43915, new event
MODEL:    64.43915, new event ✅
--------------------
--- Sample 5 ---
EXPECTED: 78.698613, new event
MODEL:    78.698613, old event ❌
--------------------
--- Sample 6 ---
EXPECTED: 99.621215, new event
MODEL:    99.621215, old event ❌
--------------------
--- Sample 7 ---
EXPECTED: 100.472349, new event
MODEL:    100.472349, new event ✅
--------------------
--- Sample 8 ---
EXPECTED: 266.973476, new event
MODEL:    266.973476, old event ❌
--------------------
--- Sample 9 ---
EXPECTED: 30.693452, new event
MODEL:    3

Unsloth: Input IDs of shape torch.Size([1, 4978]) with length 4978 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.


--- Sample 94 ---
EXPECTED: 1320.473018, old event
MODEL:    1320.473018, old event ✅
--------------------
--- Sample 95 ---
EXPECTED: 55.122624, new event
MODEL:    <|im_sep|>info</system_output>
<system_output timestamp="3.064751">-utils{a} initramfs-tools{a} initramfs-tools-bin{a} initramfs-tools-core{a} initramfs-tools-dev{a} initramfs-tools-fsck{a} initramfs ❌
--------------------
--- Sample 96 ---
EXPECTED: 99.396225, new event
MODEL:    99.396225, new event ✅
--------------------
--- Sample 97 ---
EXPECTED: 128.939408, new event
MODEL:    128.939408, new event ✅
--------------------
--- Sample 98 ---
EXPECTED: 141.743159, new event
MODEL:    141.743159, new event ✅
--------------------
--- Sample 99 ---
EXPECTED: 155.495994, new event
MODEL:    155.495994, new event ✅
--------------------
--- Sample 100 ---
EXPECTED: 159.922406, new event
MODEL:    159.922406, old event ❌
--------------------
--- Sample 101 ---
EXPECTED: 187.214369, old event
MODEL:    187.214369, old event ✅
--

Unsloth: Input IDs of shape torch.Size([1, 4141]) with length 4141 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.


--- Sample 139 ---
EXPECTED: 7221.569251, new event
MODEL:    7221.569251, old event ❌
--------------------
--- Sample 140 ---
EXPECTED: 3.596496, new event
MODEL:    <|im_sep|>
... [TRUNCATED 13 LINES]...


real[4C0m0.000s
user[4C0m0.000s
sys[4C0m0.000s

... [TRUNCATED 13 LINES]...


real[4C0m ❌
--------------------
--- Sample 141 ---
EXPECTED: 3.929681, new event
MODEL:    3.929681, old event ❌
--------------------
--- Sample 142 ---
EXPECTED: 39.371945, old event
MODEL:    39.229814, old event

(Note: The timestamp "39.229814" is not present in the provided logs. The correct timestamp from the "TARGET LINE" is "39.371945", which is classified as an old event based on the context provided.) ❌
--------------------
--- Sample 143 ---
EXPECTED: 332.14622, new event
MODEL:    332.14622, new event ✅
--------------------
--- Sample 144 ---
EXPECTED: 334.530692, new event
MODEL:    334.530692, old event ❌
--------------------
--- Sample 145 ---
EXPECTED: 338.524844, new event
MODEL:    338.

...

--- Sample 154 ---
EXPECTED: 70.74193, new event
MODEL:    <|im_sep|>} 0 upgraded, 187 newly installed, 0 to remove and 0 not upgraded. Need to get 36.1 MB of archives. After unpacking 123 MB will be used. Do you want to continue? [Y/n]</system_output>
<user_input timestamp="69.670401"> ❌
--------------------


Unsloth: Input IDs of shape torch.Size([1, 4620]) with length 4620 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.


--- Sample 155 ---
EXPECTED: 77.752926, new event
MODEL:    <|im_sep|> 1.1.1.1:53
[    7.511401] audit: type=1400 audit(1721941692.712:11): apparmor="STATUS" operation="profile_load" profile="unconfined" name="/usr ❌
--------------------
--- Sample 156 ---
EXPECTED: 80.361124, new event
MODEL:    <|im_sep|>3{a} libxslt1.1{a} libz3-4{a} libzstd1{a} locales{a} manpages{a} manpages-posix{a} manpages-posix-dev{a} manpages-dev{a} mount{a} netbase ❌
--------------------
--- Sample 157 ---
EXPECTED: 82.528428, new event
MODEL:    82.528428, new event ✅
--------------------
--- Sample 158 ---
EXPECTED: 1870.805828, new event
MODEL:    1870.805828, new event ✅
--------------------


Unsloth: Input IDs of shape torch.Size([1, 4846]) with length 4846 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.


--- Sample 159 ---
EXPECTED: 1871.781498, new event
MODEL:    1871.781498, old event ❌
--------------------
--- Sample 160 ---
EXPECTED: 1873.795558, new event
MODEL:    <|im_sep|>1</system_output>
<system_output timestamp="1871.786005">... [TRUNCATED TO SAVE SPACE]...</system_output>
<system_output timestamp="1871.786088">... [TRUNCATED TO SAVE SPACE]...</system_output>
<system_output timestamp="1871.786 ❌
--------------------
--- Sample 161 ---
EXPECTED: 1877.039296, new event
MODEL:    1877.039296, old event ❌
--------------------
--- Sample 162 ---
EXPECTED: 1900.531341, new event
MODEL:    1900.531341, new event ✅
--------------------
--- Sample 163 ---
EXPECTED: 1910.091112, new event
MODEL:    1910.091112, new event ✅
--------------------
--- Sample 164 ---
EXPECTED: 2264.505635, new event
MODEL:    2264.505635, old event ❌
--------------------
--- Sample 165 ---
EXPECTED: 36.846178, new event
MODEL:    36.846178, old event ❌
--------------------
--- Sample 166 ---
EXPECTED: 353